# Train a machine fault classifier and export it to ONNX

This is the notebook behind `fault_classifier.onnx`. Read it top to bottom, or run it in an Azure Machine Learning notebook.

1. Upload `sensor_history.csv` (from `resources/data`) next to this notebook.
2. Run all cells.
3. Download `fault_classifier.onnx` and `fault_labels.json` and copy them to the edge device.

In [ ]:
%pip install --quiet scikit-learn skl2onnx onnxruntime pandas

## 1. Look at the data

Each row is one sensor snapshot from a machine, labelled with the fault that was found afterwards.

In [ ]:
import pandas as pd

data = pd.read_csv("sensor_history.csv")
print(data.shape)
print(data["label"].value_counts())
data.head()

## 2. Train

A random forest is small, fast and needs no GPU, which makes it a good fit for an edge device.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

FEATURES = ["vibration_mm_s", "temperature_c", "load_pct", "current_a", "hours_since_service"]
LABELS = ["normal", "bearing_wear", "overheating", "low_lubrication"]

x = data[FEATURES].to_numpy(dtype="float32")
y = data["label"].map(LABELS.index).to_numpy()
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=1, stratify=y)

model = RandomForestClassifier(n_estimators=40, max_depth=6, random_state=1).fit(x_train, y_train)
pred = model.predict(x_test)
print("accuracy:", round(accuracy_score(y_test, pred), 3))
print(LABELS)
print(confusion_matrix(y_test, pred))

## 3. Export to ONNX

ONNX is a portable model format. The edge device only needs ONNX Runtime, not scikit-learn and not the cloud.

In [ ]:
import json
from skl2onnx import to_onnx

onnx_model = to_onnx(model, x[:1], options={id(model): {"zipmap": False}}, target_opset=17)
with open("fault_classifier.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())
with open("fault_labels.json", "w") as f:
    json.dump({"features": FEATURES, "labels": LABELS}, f, indent=2)
print("saved fault_classifier.onnx and fault_labels.json")

## 4. Check that ONNX Runtime gives the same answers

This is exactly what runs on the edge device: no scikit-learn, no network.

In [ ]:
import numpy as np
import onnxruntime as ort

session = ort.InferenceSession("fault_classifier.onnx", providers=["CPUExecutionProvider"])
onnx_pred = session.run(None, {session.get_inputs()[0].name: x_test})[0]
print("ONNX matches scikit-learn on", int((onnx_pred == pred).mean() * 100), "percent of the test rows")

# A machine with high vibration, high temperature and many hours since service
sample = np.array([[9.4, 78.0, 82, 31.5, 1840]], dtype="float32")
print("sample ->", LABELS[int(session.run(None, {session.get_inputs()[0].name: sample})[0][0])])